# 결정 그래프 표현 실습

**Crystal Graph · CGCNN · 구조 그래프**

결정 구조를 원자 노드와 이웃 결합 간선의 그래프로 바꾸어 모델에 넣는 표현.

소재 분야에서 이해하기: 컷오프 반경 안의 이웃을 간선으로 연결해 구조를 표현한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [소재 발견용 그래프 신경망 연구](https://www.nature.com/articles/s41586-023-06735-9)

## 1. 구조를 그래프로 바꾸기

컷오프 반경 안의 이웃을 간선으로 연결해 결정 구조를 그래프로 표현합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

def structure(kind, repeat=3, a=1.0):
    basis = {'sc': [(0, 0, 0)], 'bcc': [(0, 0, 0), (.5, .5, .5)],
             'fcc': [(0, 0, 0), (.5, .5, 0), (.5, 0, .5), (0, .5, .5)]}[kind]
    return np.array([((i + bx) * a, (j + by) * a, (k + bz) * a)
                     for i in range(repeat) for j in range(repeat) for k in range(repeat)
                     for bx, by, bz in basis])

def to_graph(points, cutoff):
    from scipy.spatial import cKDTree
    pairs = cKDTree(points).query_pairs(cutoff, output_type='ndarray')
    adjacency = np.zeros((len(points), len(points)))
    adjacency[pairs[:, 0], pairs[:, 1]] = adjacency[pairs[:, 1], pairs[:, 0]] = 1
    return adjacency, pairs

for kind in ('sc', 'bcc', 'fcc'):
    points = structure(kind, repeat=4)
    adjacency, pairs = to_graph(points, cutoff={'sc': 1.05, 'bcc': 0.9, 'fcc': 0.75}[kind])
    inner = np.all((points > 1.0) & (points < 2.5), axis=1)
    print('%s: 간선 %5d개, 내부 원자의 평균 이웃 수 %.1f'
          % (kind, len(pairs), adjacency[inner].sum(1).mean()))

## 2. 컷오프 반경이 그래프를 결정합니다

In [ ]:
points = structure('fcc', repeat=4)
inner = np.all((points > 1.0) & (points < 2.5), axis=1)
radii = np.linspace(0.6, 1.6, 30)
neighbours = []
for cutoff in radii:
    adjacency, _ = to_graph(points, cutoff)
    neighbours.append(adjacency[inner].sum(1).mean())
plt.plot(radii, neighbours, 'o-')
plt.xlabel('cutoff radius'); plt.ylabel('mean neighbours (inner atoms)'); plt.show()
print('계단이 생기는 지점이 배위 껍질(coordination shell)입니다.')
print('컷오프를 너무 작게 잡으면 구조 정보를 잃고, 너무 크게 잡으면 계산량이 급증합니다.')

## 3. 그래프에서 구조 지문 만들기

간선 길이 분포와 이웃 수 분포만으로도 구조를 구분할 수 있습니다.

In [ ]:
for kind in ('sc', 'bcc', 'fcc'):
    points = structure(kind, repeat=4)
    adjacency, pairs = to_graph(points, 1.2)
    lengths = np.linalg.norm(points[pairs[:, 0]] - points[pairs[:, 1]], axis=1)
    plt.hist(lengths, bins=40, alpha=0.5, label=kind)
    print('%s: 간선 길이 종류 %s' % (kind, np.round(np.unique(np.round(lengths, 3))[:4], 3)))
plt.xlabel('edge length'); plt.ylabel('count'); plt.legend(); plt.show()
print('\nGNN 은 이 그래프 위에서 이웃 정보를 모아 물성을 예측합니다(그래프 신경망 노트북 참고).')

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#crystal-graph)을 여세요.